# ⚡ VoiceStudio × Colab — One-Tap (GPU T4)

Semua otomatis. **3 langkah:**

1. Menu bar → **Runtime → Change runtime type → T4 GPU** → Save.
2. **Run all** (Ctrl+F9). Tunggu sel terakhir (pertama kali ± 8–12 menit).
3. Sel terakhir mencetak **ALAMAT (URL publik)** + **PIN**.
   Di app VoiceStudio di HP: **Settings → Connect / tambah komputer** → isi keduanya.
   Lalu muncul opsi **"Run on: My computer"** → kloning/generate jalan di GPU T4.

📨 Mau URL+PIN datang sendiri ke Telegram tiap sesi? Setup sekali (2 menit) — lihat 2 sel terbawah.

⚠️ URL+PIN = kunci penuh ke sesi ini — jangan dibagikan ke siapa pun.
⚠️ Sesi gratis Colab: maks ±12 jam (sering lebih pendek). Kalau putus → buka lagi, Run all, URL baru → pair ulang di app.
⚠️ Suara & audio di sesi ini hilang saat sesi didaur ulang. Mau awet antar sesi? pakai sel Opsional (Google Drive) sebelum sel 5.


In [ ]:
# 1. Cek GPU
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print("GPU terdeteksi — lanjut.")
else:
    print("=" * 74)
    print("PERINGATAN: tidak ada GPU di runtime ini.")
    print("Perbaikan: Runtime -> Change runtime type -> T4 GPU, lalu Run all dari awal.")
    print("(Tetap bisa jalan di CPU, tapi lambat.)")
    print("=" * 74)


In [ ]:
# 2. Clone + install (idempotent; once ~5-8 min)
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/debpalash/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
BUN = "/root/.bun/bin/bun"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}", flush=True)
    rc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode
    if rc != 0:
        raise SystemExit(f"\nFAILED: {what or shown} (exit {rc}). Re-run sel ini aman.")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo sudah ada di {REPO_DIR} — update.")
    before = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout
    run(["git", "-C", REPO_DIR, "pull", "--ff-only"], what="git pull")
    after = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout
    if before != after:
        shutil.rmtree(os.path.join(REPO_DIR, "frontend", "dist"), ignore_errors=True)
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], what="git clone")

run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")
if not os.path.exists(BUN):
    run("curl -fsSL https://bun.sh/install | bash", what="bun installer")
os.environ["PATH"] = os.path.dirname(BUN) + os.pathsep + os.environ["PATH"]
run([BUN, "--version"], what="bun version check")

dist_index = os.path.join(REPO_DIR, "frontend", "dist", "index.html")
if os.path.exists(dist_index):
    print("frontend/dist sudah ada — skip build.")
else:
    run([BUN, "install", "--frozen-lockfile"], cwd=REPO_DIR, what="bun install (workspace)")
    run([BUN, "run", "--cwd", "electron", "build:web"], cwd=REPO_DIR, what="electron web renderer build")
    if not os.path.exists(dist_index):
        raise SystemExit("frontend/dist/index.html tidak ada — cek error vite di atas.")

if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
run(["uv", "pip", "install", "--system", "--no-cache",
     "--constraint", "deploy/torch-constraints.txt",
     "--override", "deploy/uv-overrides.txt", "."],
    cwd=REPO_DIR, what="backend install (uv)")

os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)
run([sys.executable, os.path.join(REPO_DIR, "scripts", "setup.py")], what="cuDNN 8 compat setup")

run([sys.executable, "-c",
     "import torch, torchaudio, uvicorn, fastapi, transformers; "
     "print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, transformers {transformers.__version__}, CUDA: {torch.cuda.is_available()}'); "
     "from omnivoice.models.omnivoice import OmniVoice; "
     "from transformers import HiggsAudioV2TokenizerModel; "
     "print('Install OK - backend model stack imports cleanly')"],
    cwd=REPO_DIR, what="import sanity check")
print("\nInstall selesai.")


In [ ]:
# 3. Unduh model bawaan di depan (sekali per sesi; aman diulang)
from huggingface_hub import snapshot_download

print("TTS default (k2-fsa/OmniVoice, beberapa GB)...")
p1 = snapshot_download("k2-fsa/OmniVoice")
print("  OK:", p1)

print("ASR whisper-large-v3 (Systran/faster-whisper-large-v3, ~3 GB)...")
p2 = snapshot_download("Systran/faster-whisper-large-v3")
print("  OK:", p2)
print("\nModel siap.")


In [ ]:
# (OPSIONAL) Agar voices/audio awet antar sesi — Google Drive.
# Cara pakai: hapus tanda # pada 3 baris kode di bawah, RUN sel ini,
# lalu lanjut sel 5. Saat "Run all" biasa, sel ini tidak melakukan apa-apa.
#
# from google.colab import drive
# drive.mount('/content/drive')
# import os; os.environ['OMNIVOICE_DATA_DIR'] = '/content/drive/MyDrive/omnivoice_data'


In [ ]:
# 5. Nyalakan semuanya: tunnel publik -> backend -> PIN -> URL siap pakai
import json, os, re, subprocess, sys, time, urllib.request
from urllib.parse import urlsplit

PORT = 3900
REPO_DIR = "/content/VoiceStudio"
LOG = "/content/omnivoice_backend.log"
CF_LOG = "/content/cloudflared.log"

def sh(cmd):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def tunnel_url():
    if not os.path.exists(CF_LOG):
        return None
    m = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", open(CF_LOG, errors="replace").read())
    return m.group(1) if m else None

def cf_alive():
    return subprocess.run("pgrep -f 'cloudflared tunnel' >/dev/null", shell=True).returncode == 0

def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

# 5a. Tunnel publik
url = tunnel_url() if cf_alive() else None
if url:
    print("Tunnel sudah jalan — dipakai ulang:", url)
else:
    sh("pkill -f 'cloudflared tunnel' || true")
    time.sleep(1)
    if not os.path.exists("/usr/local/bin/cloudflared"):
        sh("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")
    open(CF_LOG, "w").close()
    subprocess.Popen("nohup cloudflared tunnel --url http://127.0.0.1:3900 --no-autoupdate >> /content/cloudflared.log 2>&1 &", shell=True)
    for _ in range(45):
        time.sleep(2)
        url = tunnel_url()
        if url:
            break
if not url:
    raise SystemExit("cloudflared belum memberi URL — tunggu 10 detik lalu re-run sel ini (log: /content/cloudflared.log).")
HOST = urlsplit(url).hostname
print("Tunnel:", url)

# 5b. Backend (server mode; izinkan host tunnel + proxy Colab)
stamp = "/content/vs_launch_host.txt"
need = (health() is None) or (not os.path.exists(stamp)) or (open(stamp).read().strip() != HOST)
if need:
    sh(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true")
    time.sleep(1)
    env = os.environ.copy()
    env["OMNIVOICE_SERVER_MODE"] = "1"
    env.setdefault("OMNIVOICE_DATA_DIR", "/content/omnivoice_data")
    env["PYTHONUNBUFFERED"] = "1"
    hosts, origins = [HOST], [f"https://{HOST}"]
    try:
        from google.colab.output import eval_js
        p = urlsplit(eval_js(f"google.colab.kernel.proxyPort({PORT})"))
        if p.hostname:
            hosts.append(p.hostname)
            origins.append(f"https://{p.hostname}")
    except Exception:
        pass
    env["OMNIVOICE_ALLOWED_HOSTS"] = ",".join(hosts)
    env["OMNIVOICE_ALLOWED_ORIGINS"] = ",".join(origins)
    logf = open(LOG, "ab")
    proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "main:app",
                             "--app-dir", "backend", "--host", "127.0.0.1", "--port", str(PORT)],
                            cwd=REPO_DIR, env=env, stdout=logf, stderr=subprocess.STDOUT)
    deadline = time.time() + 300
    while time.time() < deadline and proc.poll() is None and health() is None:
        print(".", end="", flush=True)
        time.sleep(3)
    print()
    if health() is None:
        tail = ""
        if os.path.exists(LOG):
            tail = "".join(open(LOG, errors="replace").readlines()[-25:])
        raise SystemExit("Backend tidak sehat dalam 5 menit. Log terakhir:\n" + tail)
    open(stamp, "w").write(HOST)
print("Backend OK:", health())

# 5c. Aktifkan sharing -> PIN (loopback)
req = urllib.request.Request(f"http://127.0.0.1:{PORT}/system/network/enable", method="POST")
with urllib.request.urlopen(req, timeout=60) as r:
    share = json.load(r)
PIN = str(share.get("pin") or "")
open("/content/vs_pin.txt", "w").write(PIN)
open("/content/vs_public_url.txt", "w").write(url)

print()
print("=" * 68)
print("  SIAP DIPAKAI — masukkan ini di app VoiceStudio (HP):")
print("    Alamat :", url)
print("    PIN    :", PIN)
print()
print("  App: Settings -> Connect / tambah komputer -> isi alamat + PIN.")
print("  Lalu pilih 'Run on: My computer' saat kloning / generate.")
print("=" * 68)
print("Catatan: URL+PIN = akses penuh ke sesi ini. Jangan dibagikan.")
print("Kalau sudah di-setup, sel Telegram di bawah mengirim ini ke chat-mu.")


## 📨 Kirim otomatis ke Telegram (opsional — setup sekali)

Mau URL+PIN datang sendiri ke Telegram tiap sesi? **Sekali saja:**

1. Buka Telegram → chat **@BotFather** → kirim `/newbot` → ikuti langkahnya → salin **token** yang diberi.
2. Buka bot barumu (username yang tadi kamu pilih), kirim **`/start`**.
3. Di Colab: klik ikon **🔑 (Secrets)** di sidebar kiri → **Add new secret**:
   - **Name**: `TELEGRAM_BOT_TOKEN`
   - **Value**: token dari BotFather
   - toggle **"Notebook access"** → ON → **Save**.
4. Selesai. Tiap sesi berikutnya, sel di bawah otomatis mengirim URL+PIN ke chat bot itu.

⚠️ Token = kunci bot-mu. Tempel **hanya** di kolom Secret Colab — jangan di chat mana pun (termasuk chat Sipinter).


In [ ]:
# 7. Kirim URL + PIN ke Telegram kamu (jalan hanya jika sudah di-setup; kalau belum, dilewati)
import json, os, re, urllib.request

def tg_call(token, method, params=None):
    data = json.dumps(params or {}).encode()
    req = urllib.request.Request(f"https://api.telegram.org/bot{token}/{method}",
                                 data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.load(r)

def secret(name):
    v = os.environ.get(name, "").strip()
    if v:
        return v
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

token = secret("TELEGRAM_BOT_TOKEN")
if not token:
    print("TELEGRAM_BOT_TOKEN belum di-set -> sel ini dilewati.")
    print("Setup ada di sel markdown di atas (BotFather + Colab Secrets).")
else:
    url = None
    try:
        url = open("/content/vs_public_url.txt").read().strip() or None
    except OSError:
        pass
    if not url and os.path.exists("/content/cloudflared.log"):
        m = re.search(r"(https://[a-z0-9-]+\.trycloudflare\.com)", open("/content/cloudflared.log", errors="replace").read())
        url = m.group(1) if m else None
    pin = None
    try:
        pin = open("/content/vs_pin.txt").read().strip() or None
    except OSError:
        pass

    if not url or not pin:
        print("URL/PIN belum ada — jalankan dulu sel 5 (Nyala semuanya), lalu sel ini.")
    else:
        chat = secret("TELEGRAM_CHAT_ID")
        if not chat:
            try:
                ups = tg_call(token, "getUpdates", {"timeout": 0, "limit": 20})
                cands = [(u.get("message") or {}).get("chat") or {} for u in ups.get("result", [])]
                cands = [c for c in cands if c.get("id")]
                if cands:
                    chat = str(cands[-1]["id"])
            except Exception as e:
                print("getUpdates gagal:", e)
        if not chat:
            print("Chat belum ketemu — kirim /start ke bot barumu sekali, lalu jalankan ulang sel ini.")
        else:
            text = ("⚡ VoiceStudio Colab siap!\n\n"
                    "Alamat:\n" + url + "\n\n"
                    "PIN: " + pin + "\n\n"
                    "Di app: Settings → Connect → isi alamat & PIN.\n"
                    "Jangan bagikan pesan ini (URL+PIN = akses penuh).")
            try:
                res = tg_call(token, "sendMessage", {"chat_id": chat, "text": text, "disable_web_page_preview": True})
                print("Terkirim ke Telegram ✓" if res.get("ok") else f"Telegram menolak: {res}")
            except Exception as e:
                print("Gagal kirim:", e)


## Setelah jalan

- **Generate pertama** di sesi ini agak lama (model dimuat); berikutnya hitungan detik.
- URL & PIN hanya untuk sesi ini. Sesi baru / habis putus → **Run all** lagi → URL baru → pair ulang di app.
- Bot Telegram belum kirim? Cek: secret `TELEGRAM_BOT_TOKEN` ada & "Notebook access" ON, dan kamu sudah kirim `/start` ke bot itu.
- Sesi gratis: maks ±12 jam (sering lebih pendek); semua di `/content` hilang saat VM didaur ulang (aktifkan sel Opsional Drive kalau mau awet).
- Ada sel yang error? Screenshot sel merahnya dan kirim ke Sipinter.
- App tetap bisa mode on-device ("This phone") kapan saja sebagai cadangan tanpa Colab.
